In [1]:
import polars as pl
from src.load_data import load_main_data

In [3]:
df = load_main_data.load_dev_data()
df.head(1)

session_id,user_id,session_date,user_profile,conversation_goal,conversations,goal_progress_assessments,split_dev
str,str,str,struct[9],struct[3],list[struct[4]],list[struct[2]],str
"""9c337a02-15b1-408f-8103-c2f945…","""64ea97af-bbc6-4756-ac94-b93104…","""2011-12-26""","{20,""20s"",""BR"",""Brazil"",""female"",""English"",""Western Alternative Rock"",""64ea97af-bbc6-4756-ac94-b931048e5fef"",""train_warm""}","{""H"",""The listener wants to explore different artists and discover new songs from a broad collection, focusing on general moods and styles, without specific genre constraints. They are open to exploring the discographies of artists they find interesting."",""LL""}","[{""I want to discover some new artists. Do you have anything that's a bit intense or dramatic?"",""user"","""",1}, {""81d9f1d9-3b22-4836-9f06-2140e959e6de"",""music"",""You enjoyed the last Alesana track, so I'm sticking with your favorite artist and genre. ""The Fiend"" carries that same intense post-hardcore energy and screamo vocals that define Alesana's sound."",1}, … {""Perfect! Glad to hear it. Let's keep exploring that album with ""The Best Laid Plans Of Mice And Marionettes."" It’s another great one from Alesana with all the post-hardcore goodness you're into."",""assistant"","""",8}]","[{null,1}, {""MOVES_TOWARD_GOAL"",2}, … {""DOES_NOT_MOVE_TOWARD_GOAL"",8}]","""train"""


In [23]:
def preprocess(df: pl.DataFrame) -> pl.DataFrame:
    df = (
        df
        .explode("conversations")
        .unnest("conversations")
        .drop("user_id")
        .unnest("user_profile")
    )

    df_user = df.filter(pl.col("role") == "user")

    # ターゲットの抽出
    df_music = (
        df
        .filter(pl.col("role") == "music")
        .select(["session_id", "user_id", "turn_number", "content"])
        .rename({"content": "ans_track_id"})
    )

    ret_df = df_user.join(df_music, on=["session_id", "user_id", "turn_number"])

    return ret_df

df_processed = preprocess(df)

In [24]:
df_processed.head(1)

session_id,session_date,age,age_group,country_code,country_name,gender,preferred_language,preferred_musical_culture,user_id,user_split,conversation_goal,content,role,thought,turn_number,goal_progress_assessments,split_dev,ans_track_id
str,str,i64,str,str,str,str,str,str,str,str,struct[3],str,str,str,i64,list[struct[2]],str,str
"""9c337a02-15b1-408f-8103-c2f945…","""2011-12-26""",20,"""20s""","""BR""","""Brazil""","""female""","""English""","""Western Alternative Rock""","""64ea97af-bbc6-4756-ac94-b93104…","""train_warm""","{""H"",""The listener wants to explore different artists and discover new songs from a broad collection, focusing on general moods and styles, without specific genre constraints. They are open to exploring the discographies of artists they find interesting."",""LL""}","""I want to discover some new ar…","""user""","""""",1,"[{null,1}, {""MOVES_TOWARD_GOAL"",2}, … {""DOES_NOT_MOVE_TOWARD_GOAL"",8}]","""train""","""81d9f1d9-3b22-4836-9f06-2140e9…"
